# 🤖 Fine-Tuning GPT-2 for Real-World Applications

This lab fine-tunes a pre-trained GPT-2 model for two industry use cases:

| Component | Domain | Task |
|-----------|--------|------|
| **I** | E-Commerce | Product Review Generator |
| **II** | Food-Tech | Recipe Instruction Generator |

**Learning Outcomes:**
1. Understand how fine-tuning applies to real-world industry applications
2. Load and configure a pre-trained GPT-2 model using Hugging Face Transformers
3. Prepare domain-specific datasets for causal language modeling
4. Fine-tune the model and compare output before and after training
5. Evaluate quality using perplexity

> ⚡ **Tip:** Go to **Runtime → Change runtime type → T4 GPU** for faster training.

---
## ⚙️ Setup — Install Dependencies

In [1]:
!pip install transformers datasets accelerate -q

In [2]:
import torch
import math
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
    set_seed
)
from datasets import Dataset

set_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

Using device: cuda


---
## 🛠️ Shared Utility — Text Generation Function

This function is reused across both components to generate text from any GPT-2 model.

In [3]:
def generate_text(model, tokenizer, prompt, max_length=100):
    """Generate text continuation from a given prompt."""
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        output = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

---
# 📦 Component I — Product Review Generator (E-Commerce)

**Scenario:** You are an AI engineer at an e-commerce company. Fine-tune GPT-2 on real product review data so the model learns the style, vocabulary, and tone of consumer reviews.

**Tasks:**
1. Load GPT-2 and generate baseline reviews (before fine-tuning)
2. Prepare the product review dataset and tokenize it
3. Fine-tune the model on product review data
4. Generate reviews from the fine-tuned model and compare

### Step 1 — Load Model

In [4]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print('GPT-2 loaded successfully.')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 loaded successfully.
Model parameters: 124,439,808


### Step 2 — Baseline Reviews (Before Fine-Tuning)

In [5]:
review_prompts = [
    'This product is',
    'I bought this phone and',
    'The quality of this item',
]

print('=' * 60)
print('BASELINE REVIEWS (Before Fine-Tuning)')
print('=' * 60)

baseline = {}
for prompt in review_prompts:
    baseline[prompt] = generate_text(model, tokenizer, prompt)
    print(f'\nPrompt : {prompt}')
    print(f'Output : {baseline[prompt]}')
    print('-' * 60)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


BASELINE REVIEWS (Before Fine-Tuning)

Prompt : This product is
Output : This product is made from high quality, lightweight stainless steel. If you are looking for something a little more durable, it's a good choice.

Laser Pouch

Not all of our laser printers are created equal. We have a laser printer that comes with all of our printer parts. These parts include our new 3D printer and a 3D printed printing service. All of our printers make laser printers, including our laser printers, using laser technology. Our laser printers are the most
------------------------------------------------------------

Prompt : I bought this phone and
Output : I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen was amazing and the sound was amazing. It was not loud. I would never use it on a tv, laptop, smartphone or other connected device in the future.

The battery life is good. The phone works great but it has so many problems.

I h

### Step 3 — Prepare Dataset and Fine-Tune

In [6]:
corpus = [
    'this phone has an amazing battery life and the camera quality is outstanding for the price.',
    'i bought this laptop for college and it handles all my assignments and coding projects perfectly.',
    'the sound quality of these headphones is incredible with deep bass and clear vocals.',
    'this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.',
    'great wireless earbuds with noise cancellation that blocks out all background sound.',
    'the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.',
    'this portable charger saved me during travel and it charges my phone three times on a single charge.',
    'the tablet screen is bright and colorful which makes watching movies a great experience.',
    'i love this fitness tracker because it motivates me to reach my daily exercise goals.',
    'this bluetooth speaker is compact but delivers surprisingly loud and clear audio.',
    'the delivery was fast and the product was packed securely with no damage at all.',
    'excellent value for money and the build quality feels premium despite the affordable price.',
    'the customer service team was very helpful when i had questions about the product features.',
    'this camera takes stunning photos in low light and the video recording quality is very smooth.',
    'i have been using this product for three months and it still works perfectly like day one.',
    'the design is sleek and modern and it looks great on my desk next to my other gadgets.',
    'easy to set up right out of the box and the instructions were clear and simple to follow.',
    'highly recommend this product to anyone looking for quality and reliability at a fair price.',
    'the software updates keep adding new features which makes this purchase even more worthwhile.',
    'best purchase i made this year and i would definitely buy from this brand again.',
]

# Build dataset
dataset = Dataset.from_dict({'text': corpus})
tokenized = dataset.map(
    lambda x: tokenizer(x['text'], truncation=True, max_length=128, padding='max_length'),
    batched=True,
    remove_columns=['text']
)
split = tokenized.train_test_split(test_size=0.15, seed=42)

print(f'Train samples : {len(split["train"])}')
print(f'Test samples  : {len(split["test"])}')

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Train samples : 17
Test samples  : 3


In [7]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir='./gpt2-reviews',
    num_train_epochs=15,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy='epoch',
    logging_steps=10,
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    data_collator=data_collator,
)

print('Starting fine-tuning for Component I...')
trainer.train()

Starting fine-tuning for Component I...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,No log,3.348063
2,4.057265,3.247350
3,4.057265,3.106139
4,3.319935,2.981021
5,3.319935,2.881039
6,2.706133,2.803390
7,2.706133,2.756230
8,1.918991,2.740822
9,1.918991,2.765902
10,1.234805,2.867215


TrainOutput(global_step=75, training_loss=1.9250287294387818, metrics={'train_runtime': 12.8059, 'train_samples_per_second': 19.913, 'train_steps_per_second': 5.857, 'total_flos': 16657367040000.0, 'train_loss': 1.9250287294387818, 'epoch': 15.0})

### Step 4 — Evaluate and Compare

In [8]:
eval_results = trainer.evaluate()
perplexity = math.exp(eval_results['eval_loss'])
print(f'Perplexity (Component I): {perplexity:.2f}')
print('(Lower perplexity = model is less surprised by domain text)')

Perplexity (Component I): 24.74
(Lower perplexity = model is less surprised by domain text)


In [9]:
print('=' * 60)
print('COMPARISON: Baseline vs Fine-Tuned (Product Reviews)')
print('=' * 60)

for prompt in review_prompts:
    ft_output = generate_text(model, tokenizer, prompt)
    print(f'\nPrompt     : {prompt}')
    print(f'Baseline   : {baseline[prompt][:150]}')
    print(f'Fine-Tuned : {ft_output[:150]}')
    print('-' * 60)

COMPARISON: Baseline vs Fine-Tuned (Product Reviews)

Prompt     : This product is
Baseline   : This product is made from high quality, lightweight stainless steel. If you are looking for something a little more durable, it's a good choice.

Lase
Fine-Tuned : This product is packed with features that make this purchase even more worthwhile. The quality of this product exceeds expectations and the price poin
------------------------------------------------------------

Prompt     : I bought this phone and
Baseline   : I bought this phone and I have not used it on a lot of people. I have also not used it on any other people.

The screen was amazing and the sound was 
Fine-Tuned : I bought this phone and it handles all my daily chores perfectly. I would definitely buy from this brand again.

Verified purchase: this deal i made t
------------------------------------------------------------

Prompt     : The quality of this item
Baseline   : The quality of this item in the item description 

**Expected observations:**
- Baseline output: generic, off-topic (news/Wikipedia style)
- Fine-tuned output: product-domain language — *"great value"*, *"highly recommend"*, *"battery life"*, *"build quality"*

---
# 🍳 Component II — Recipe Instruction Generator (Food-Tech)

**Scenario:** You are an AI developer at a food-tech startup building a smart cooking app. Fine-tune GPT-2 to generate step-by-step cooking instructions when given a dish name as a prompt.

**Tasks:**
1. Reload a fresh GPT-2 (independent of Component I)
2. Prepare the recipe instruction dataset and tokenize it
3. Fine-tune the model on recipe data
4. Generate recipe instructions and compare with baseline

### Step 1 — Reload Fresh Model

In [10]:
tokenizer2 = GPT2Tokenizer.from_pretrained('gpt2')
model2 = GPT2LMHeadModel.from_pretrained('gpt2')

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

print('Fresh GPT-2 loaded for Component II.')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fresh GPT-2 loaded for Component II.


### Step 2 — Baseline Recipes (Before Fine-Tuning)

In [11]:
recipe_prompts = [
    'To make butter chicken',
    'For pasta carbonara',
    'To prepare a chocolate cake',
]

print('=' * 60)
print('BASELINE RECIPES (Before Fine-Tuning)')
print('=' * 60)

baseline2 = {}
for prompt in recipe_prompts:
    baseline2[prompt] = generate_text(model2, tokenizer2, prompt)
    print(f'\nPrompt : {prompt}')
    print(f'Output : {baseline2[prompt]}')
    print('-' * 60)

BASELINE RECIPES (Before Fine-Tuning)

Prompt : To make butter chicken
Output : To make butter chicken with eggs. I was going to do this with a different recipe, but I was going into a different area of town and was going to have to do something with them. I love a good egg.

Here's how to make your own:

1 pound unsalted butter

1 tsp vanilla extract

8-10 eggs

1 cup all-purpose flour

4 tablespoons unsalted butter

1/2 cup all-purpose flour

------------------------------------------------------------

Prompt : For pasta carbonara
Output : For pasta carbonara pasta

2 large olives, cut into thin strips

1 large onion, finely chopped

4 small garlic cloves, finely chopped

3 large red bell peppers, finely chopped

1 large green chile, finely chopped

4 cups water

1 tsp dried oregano or ground oregano

salt to taste

4 cups white wine vinegar

1 tbsp olive oil

1 tsp salt

1 cup water
------------------------------------------------------------

Prompt : To prepare a chocolate cake
O

### Step 3 — Prepare Recipe Dataset and Fine-Tune

In [12]:
recipes = [
    'to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one hour.',
    'heat butter in a pan and fry onions until golden brown then add ginger garlic paste and cook for two minutes.',
    'add tomato puree and cook on low heat for ten minutes until the oil separates from the masala.',
    'add the marinated chicken and cook on medium heat for fifteen minutes until fully cooked.',
    'finish with fresh cream and kasuri methi and serve hot with naan or steamed rice.',
    'for pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water.',
    'fry diced pancetta in olive oil until crispy and set aside.',
    'whisk together egg yolks parmesan cheese and black pepper in a bowl.',
    'toss the hot pasta with pancetta and remove from heat then quickly stir in the egg mixture.',
    'the residual heat will cook the eggs into a creamy sauce and serve immediately with extra parmesan.',
    'to prepare vegetable stir fry heat sesame oil in a wok on high heat.',
    'add sliced bell peppers broccoli florets and snap peas and toss for three minutes.',
    'pour in soy sauce oyster sauce and a pinch of sugar and stir well.',
    'add minced garlic and ginger and cook for one more minute until fragrant.',
    'serve the stir fry over steamed jasmine rice and garnish with sesame seeds.',
    'for chocolate chip cookies cream together butter and sugar until light and fluffy.',
    'beat in eggs one at a time then add vanilla extract and mix well.',
    'fold in flour baking soda and salt then gently stir in chocolate chips.',
    'scoop dough onto a baking sheet and bake at 180 degrees for twelve minutes until golden.',
    'let cookies cool on the tray for five minutes before transferring to a wire rack.',
]

# Build dataset
dataset2 = Dataset.from_dict({'text': recipes})
tok2 = dataset2.map(
    lambda x: tokenizer2(x['text'], truncation=True, max_length=128, padding='max_length'),
    batched=True,
    remove_columns=['text']
)
split2 = tok2.train_test_split(test_size=0.15, seed=42)

print(f'Train samples : {len(split2["train"])}')
print(f'Test samples  : {len(split2["test"])}')

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Train samples : 17
Test samples  : 3


In [13]:
collator2 = DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)

args2 = TrainingArguments(
    output_dir='./gpt2-recipes',
    num_train_epochs=15,
    per_device_train_batch_size=4,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    eval_strategy='epoch',
    logging_steps=10,
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer2 = Trainer(
    model=model2,
    args=args2,
    train_dataset=split2['train'],
    eval_dataset=split2['test'],
    data_collator=collator2,
)

print('Starting fine-tuning for Component II...')
trainer2.train()

Starting fine-tuning for Component II...


Epoch,Training Loss,Validation Loss
1,No log,3.403592
2,3.926337,3.264174
3,3.926337,3.105294
4,3.443312,3.021800
5,3.443312,2.981154
6,2.728940,2.930787
7,2.728940,2.897742
8,2.090287,2.897580
9,2.090287,2.992910
10,1.577087,3.140069


TrainOutput(global_step=75, training_loss=2.0859599272410074, metrics={'train_runtime': 7.2089, 'train_samples_per_second': 35.373, 'train_steps_per_second': 10.404, 'total_flos': 16657367040000.0, 'train_loss': 2.0859599272410074, 'epoch': 15.0})

### Step 4 — Evaluate and Compare

In [14]:
eval2 = trainer2.evaluate()
perplexity2 = math.exp(eval2['eval_loss'])
print(f'Perplexity (Component II): {perplexity2:.2f}')

Perplexity (Component II): 36.75


In [15]:
print('=' * 60)
print('COMPARISON: Baseline vs Fine-Tuned (Recipe Instructions)')
print('=' * 60)

for prompt in recipe_prompts:
    ft = generate_text(model2, tokenizer2, prompt)
    print(f'\nPrompt     : {prompt}')
    print(f'Baseline   : {baseline2[prompt][:150]}')
    print(f'Fine-Tuned : {ft[:150]}')
    print('-' * 60)

COMPARISON: Baseline vs Fine-Tuned (Recipe Instructions)

Prompt     : To make butter chicken
Baseline   : To make butter chicken with eggs. I was going to do this with a different recipe, but I was going into a different area of town and was going to have 
Fine-Tuned : To make butter chicken add turmeric chili powder and garam masala and cook for one more minute until the chicken is soft. Add chicken pieces and cook 
------------------------------------------------------------

Prompt     : For pasta carbonara
Baseline   : For pasta carbonara pasta

2 large olives, cut into thin strips

1 large onion, finely chopped

4 small garlic cloves, finely chopped

3 large red bel
Fine-Tuned : For pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water. Drain pasta and remove from heat. Add reserved 
------------------------------------------------------------

Prompt     : To prepare a chocolate cake
Baseline   : To prepare a chocolate cake, you're goi

**Expected observations:**
- Baseline output: unrelated text for cooking prompts
- Fine-tuned output: step-by-step cooking instructions with ingredients, temperatures, and timing
- Generated recipes follow a realistic cooking flow: *marinate → cook → serve*

---
## 📊 Summary

Run the cell below to print a side-by-side summary of both components.

In [16]:
print('=' * 60)
print('LAB SUMMARY')
print('=' * 60)
print(f'Component I  — Product Review Generator')
print(f'  Perplexity : {perplexity:.2f}')
print(f'  Domain     : E-Commerce reviews')
print(f'  Dataset    : 20 product review sentences')
print()
print(f'Component II — Recipe Instruction Generator')
print(f'  Perplexity : {perplexity2:.2f}')
print(f'  Domain     : Food-Tech cooking instructions')
print(f'  Dataset    : 20 recipe instruction sentences (4 dishes)')
print()
print('Key takeaway: Both models shifted from generic web-style text')
print('to domain-specific language after fine-tuning on just 20 samples.')

LAB SUMMARY
Component I  — Product Review Generator
  Perplexity : 24.74
  Domain     : E-Commerce reviews
  Dataset    : 20 product review sentences

Component II — Recipe Instruction Generator
  Perplexity : 36.75
  Domain     : Food-Tech cooking instructions
  Dataset    : 20 recipe instruction sentences (4 dishes)

Key takeaway: Both models shifted from generic web-style text
to domain-specific language after fine-tuning on just 20 samples.
